# Randomly sampling words with torch.multinomial

In [1]:
import torch
import torch.nn.functional as F
import numpy as np

## Generate a sample from a vector

In [2]:
# a vector (must be tensor)
vect = torch.tensor([1,2,5], dtype=torch.float)

# sample a number
torch.multinomial(vect, 1)

tensor([2])

## Some errors we can encounter

In [4]:
# requires torch tensor
# torch.multinomial([1.,2.,3.], 1)
torch.multinomial(np.array([1.,2.,3.]), 1)

TypeError: multinomial(): argument 'input' (position 1) must be Tensor, not numpy.ndarray

In [5]:
# default is no replacement
torch.multinomial(vect, len(vect)+1) # use replacement=True

RuntimeError: cannot sample n_sample > prob_dist.size(-1) samples without replacement

In [7]:
# only floats
torch.multinomial(torch.tensor([1,1,1]), 1)

RuntimeError: multinomial only supports floating-point dtypes for input, got: Long

In [8]:
# only non-negative numbers
torch.multinomial(torch.tensor([-1, 1., 1]), 1)

RuntimeError: probability tensor contains either `inf`, `nan` or element < 0

## Generate many samples from the same vector

In [9]:
# sample 10 times from that vector
vect[torch.multinomial(vect, 10, replacement=True)]

tensor([2., 5., 5., 5., 5., 1., 5., 2., 5., 1.])

In [10]:
# 10k samples!
mn = torch.multinomial(vect, 10000, replacement=True)

# collect the distribution
vals, counts = np.unique(mn, return_counts=True)

# print the output values and how often they occurred
for v, c in zip(vals, counts):
    print(f'"{v}" was sampled {c} times ({c*100/len(mn):.2f}%)')

"0" was sampled 1216 times (12.16%)
"1" was sampled 2530 times (25.30%)
"2" was sampled 6254 times (62.54%)


In [11]:
# treat the vector as if it contains (scaled) probability values

# again with more information
for v, c, vectval in zip(vals, counts, vect):
    observed_frequency = c*100 / len(mn)
    expected_frequency = vectval*100 / torch.sum(vect)

    print(f'"{v}" was sampled {c} times. That is {observed_frequency:.2f}%, and the expected probability is {expected_frequency}%')

"0" was sampled 1216 times. That is 12.16%, and the expected probability is 12.5%
"1" was sampled 2530 times. That is 25.30%, and the expected probability is 25.0%
"2" was sampled 6254 times. That is 62.54%, and the expected probability is 62.5%


## with softmaxification

In [12]:
# softmax the vector
vect_softmax = F.softmax(vect, dim=-1)

# new reporting
for v, c, vectval in zip(vals, counts, vect_softmax):
    observed_frequency = c*100 / len(mn)

    print(f'"{v}" was sampled {c:4} times. That is {observed_frequency:5.2f}%, and the softmax probability is {vectval*100:5.2f}%')


"0" was sampled 1216 times. That is 12.16%, and the softmax probability is  1.71%
"1" was sampled 2530 times. That is 25.30%, and the softmax probability is  4.66%
"2" was sampled 6254 times. That is 62.54%, and the softmax probability is 93.62%


In [13]:
# now sampling from the softmax vector
mn = torch.multinomial(vect_softmax, 10000, replacement=True)
vals, counts = np.unique(mn, return_counts=True)

# new reporting
for v, c, vectval in zip(vect_softmax, counts, vect_softmax):
    observed_frequency = c*100 / len(mn)
    print(f'"{v:.4f}" was sampled {c:4} times. That is {observed_frequency:5.2f}%, and the softmax probability is {vectval*100:5.2f}%')

"0.0171" was sampled  163 times. That is  1.63%, and the softmax probability is  1.71%
"0.0466" was sampled  454 times. That is  4.54%, and the softmax probability is  4.66%
"0.9362" was sampled 9383 times. That is 93.83%, and the softmax probability is 93.62%


## comparison with numpy.random.choice

In [15]:
np.random.choice(vect, 10)

array([1., 5., 2., 1., 1., 1., 5., 5., 2., 1.], dtype=float32)

In [16]:
# sample lots of values
mn = np.random.choice(vect,10000,replace=True)

# report the results
vals,counts = np.unique(mn,return_counts=True)
for v,c,vectval in zip(vals,counts,vect):

  observed_frequency = c*100/len(mn)
  expected_frequency = 1*100/len(vect)

  print(f'"{v}" was sampled {c} times. That is {observed_frequency:.2f}%, and the expected probability is {expected_frequency:.2f}%')

"1.0" was sampled 3395 times. That is 33.95%, and the expected probability is 33.33%
"2.0" was sampled 3262 times. That is 32.62%, and the expected probability is 33.33%
"5.0" was sampled 3343 times. That is 33.43%, and the expected probability is 33.33%


In [17]:
### getting np.random.choice to match multinomial's function

# define probabilities (weights for selection)
probvalues = vect / sum(vect)

# sample lots of values
mn = np.random.choice(vect, 10000, replace=True, p=probvalues)

# report the results
vals,counts = np.unique(mn,return_counts=True)
for v,c,p in zip(vals,counts,probvalues):

  observed_frequency = c*100/len(mn)
  expected_frequency = p*100

  print(f'"{v}" was sampled {c} times. That is {observed_frequency:.2f}%, and the expected probability is {expected_frequency:.2f}%')

"1.0" was sampled 1292 times. That is 12.92%, and the expected probability is 12.50%
"2.0" was sampled 2520 times. That is 25.20%, and the expected probability is 25.00%
"5.0" was sampled 6188 times. That is 61.88%, and the expected probability is 62.50%
